### Phase 4: Backtesting-Strategie
Wir plotten zunächst die Regime-Wahrscheinlichkeiten der Modelle sowie der tatsächlichen Modell-Signale.

Anschließend testen wir den Erfolg einer Investition in Abhängigkeit zum gewählten Modell und den unten beschriebenen Regeln.

*   **Regel:**
    *   Wenn Modell sagt "Bull": 100% Aktien (S&P 500).
    *   Wenn Modell sagt "Bear": 100% Bonds oder Cash.
*   **Transaktionskosten:** Integriere realistische Kosten (z.B. 0,1% pro Trade), da ML-Modelle oft zu nervös hin- und herschalten ("Churning").

In [ ]:
# --- Central Config ---
import sys; sys.path.insert(0, "../config")
from config_loader import cfg

In [ ]:
import pandas as pd

# Daten aus dem data-Ordner laden
test_df = pd.read_parquet(cfg.data_path("test_data"))

In [ ]:
# --- 5. Dynamisches Backtesting (Automatisierter Vergleich aller Modelle) ---

import matplotlib.pyplot as plt
import numpy as np

# --- Backtesting-Engine aus src/ ---
sys.path.insert(0, "..")
from src.backtest.engine import run_all_backtests

# Parameterdefinition aus zentraler Config
transaction_costs = cfg.transaction_cost_rate  # z.B. 10 bps → 0.001

# Alle Modelle dynamisch backtesten
backtesting_results, backtesting_transaction_costs = run_all_backtests(
    test_df=test_df,
    fee_rate=transaction_costs,
    signal_shift=cfg.backtesting.signal_shift,
)

# Dynamische Visualisierung
plt.figure(figsize=(14, 7))

color_map = cfg.color_map
default_colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

plt.plot(backtesting_results['Buy_Hold'], 
         label='Statisches 60/40 Portfolio (Benchmark)', 
         color=color_map.get('Buy_Hold', 'gray'), 
         alpha=0.5, linestyle='--')

for col in backtesting_results.columns:
    if col == 'Buy_Hold':
        continue
    color = color_map.get(col, None)
    linewidth = 1.5
    alpha = 0.8
    plt.plot(backtesting_results[col], 
             label=f'Strategie: {col.replace("_", " ")}', 
             color=color, linewidth=linewidth, alpha=alpha)

plt.title("Equity Curves: Dynamischer Vergleich der Regime-Switching-Modelle")
plt.xlabel("Datum")
plt.ylabel("Kumuliertes Vermögen (Start = 1.0)")
plt.legend(loc='upper left', ncol=2)
plt.grid(True, alpha=0.2)

# Equity Curve persistieren
plt.savefig(cfg.asset_path("equity_curves"), dpi=300, bbox_inches='tight')
plt.show()

print(f"Backtesting abgeschlossen. {len([c for c in test_df.columns if c.endswith('_Signal')])} Strategien wurden berechnet.")

print("backtesting_results Dataframe")
print(backtesting_results)
print("backtesting_transaction_costs Dataframe")
print(backtesting_transaction_costs)

#--- Wir erhalten in diesem Schritt neben df und test_df backtesting_results mit kumulierten Werten ----
#--- Außerdem erhalten wir das Dataframe backtesting_transaction_costs mit Verläufen der Transaktionskosten ---

In [ ]:
# --- Performance & Drawdown Zusammenfassung ---

print("\n--- PERFORMANCE & DRAWDOWN ZUSAMMENFASSUNG ---")

# --- Performance-Summary aus src/ ---
from src.backtest.engine import calculate_performance_summary

performance_summary_df = calculate_performance_summary(backtesting_results)

# Als Markdown persistieren
performance_summary_df.to_markdown(cfg.asset_path("performance_summary"))

print("Zusammenfassung erfolgreich unter " + cfg.asset_path("performance_summary") + " gespeichert.")

# Im Notebook anzeigen
display(performance_summary_df)

In [ ]:
# Visualisierung der kumulierten Transaktionskosten
plt.figure(figsize=(14, 5))
for col in backtesting_transaction_costs.columns:
    if col == 'Buy_Hold': continue
    color = color_map.get(col, None)
    plt.plot(backtesting_transaction_costs[col] * 100, label=f'Kosten: {col.replace("_", " ")}', color=color)

plt.title(f"Kumulierte Transaktionskosten im Zeitverlauf (Gebühr: {transaction_costs*100}%)")
plt.ylabel("Kosten in %")
plt.legend(loc='upper left')
plt.grid(True, alpha=0.2)
plt.savefig(cfg.asset_path("transaction_costs"), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
#--- SORR Simulation: Analyse der Entnahmephase ---

import matplotlib.pyplot as plt

# --- SORR aus src/ ---
from src.backtest.sorr import run_sorr_simulation, build_sorr_scenarios, build_sorr_summary

# Definition der Szenarien aus zentraler Config
scenarios = build_sorr_scenarios(cfg.backtesting.sorr.scenarios)

sorr_summaries = []

# Initialisierung eines leeren DataFrames für alle Simulationspfade
backtesting_sorr_simulation = pd.DataFrame(index=backtesting_results.index)

color_map = cfg.color_map
default_colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

for name, params in scenarios.items():
    print(f"Running Scenario: {name}...")
    
    # Simulation berechnen
    sim_results = run_sorr_simulation(
        backtesting_results, 
        test_df, 
        params["start"], 
        params["withdrawal"], 
        params["fee"]
    )
    
    # Alle Ergebnisse dem zentralen DataFrame hinzufügen
    for col in sim_results.columns:
        column_name = f"{name}_{col}"
        backtesting_sorr_simulation[column_name] = sim_results[col]
    
    # Visualisierung pro Szenario
    plt.figure(figsize=(12, 5))

    for i, col in enumerate(sim_results.columns):
        strategy_name = col
        if strategy_name in color_map:
            color = color_map[strategy_name]
        else:
            color = default_colors[i % len(default_colors)]
        plt.plot(sim_results[col],
                 label=col.replace('_', ' '),
                 color=color)
    
    plt.title(f"SORR Szenario {name}: Start {params['start']:,.0f}€, Entnahme {params['withdrawal']:,.0f}€")
    plt.axhline(y=0, color='black', linestyle='-')
    plt.legend(loc='upper left', fontsize='small')
    plt.savefig(cfg.asset_path(f"sorr_sim_{name.lower()}"), dpi=300, bbox_inches='tight')
    plt.show()

    # Statistische Auswertung für dieses Szenario sammeln
    sorr_summaries.extend(build_sorr_summary(sim_results, name))

# Zusammenfassende Tabelle aller Szenarien erstellen und persistieren
sorr_multi_eval = pd.DataFrame(sorr_summaries).set_index(['Szenario', 'Strategie'])
sorr_multi_eval.to_markdown(cfg.asset_path("sorr_summary"), index=True)

print("Alle Szenarien berechnet und unter " + cfg.paths.assets_dir + "/ gespeichert.")
display(sorr_multi_eval)

print(backtesting_sorr_simulation)

#--- Wir erhalten in diesem Schritt neben df, test_df, backtesting_results, backtesting_transaction_costs nun backtesting_sorr_simulation ----

In [ ]:
from pathlib import Path

output_path = Path(cfg.data_path('backtesting_results'))
output_path.parent.mkdir(parents=True, exist_ok=True)

backtesting_results.to_parquet(output_path)
print(f"Dataframe erfolgreich unter {output_path} gespeichert.")

In [ ]:
output_path = cfg.data_path("backtesting_costs")

# Speichern als Parquet
backtesting_transaction_costs.to_parquet(output_path)

print(f"Transaktionskosten-Dataframe erfolgreich unter {output_path} gespeichert.")

In [ ]:
output_path = cfg.data_path("backtesting_sorr")

# Speichern als Parquet
backtesting_sorr_simulation.to_parquet(output_path)

print(f"SORR-Simulations-Dataframe erfolgreich unter {output_path} gespeichert.")